In [ ]:
import tensorflow as tf
import numpy as np
import os

# --------------------------------
# 1. TPU SETUP (initialize only once)
# --------------------------------
if not globals().get('TPU_INITIALIZED', False):
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()  # TPU detection
        print('Running on TPU:', tpu.master())
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.TPUStrategy(tpu)
        TPU_INITIALIZED = True  # set flag so that TPU is not reinitialized later
    except ValueError:
        print("TPU not found. Running on CPU/GPU instead.")
        strategy = tf.distribute.get_strategy()
else:
    print("TPU already initialized. Reusing the existing TPU strategy.")
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    strategy = tf.distribute.TPUStrategy(tpu)

print("Number of replicas:", strategy.num_replicas_in_sync)

# --------------------------------
# 2. DATA PREPROCESSING
# --------------------------------
data_path = "dataset.txt"
with open(data_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Length of text: {len(text)} characters")

# Create a sorted list of unique characters (vocabulary)
vocab = sorted(set(text))
vocab_size = len(vocab)
print(f"{vocab_size} unique characters")

# Create mappings from characters to indices and vice versa.
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

# Convert the entire text into integer IDs.
text_as_int = np.array([char2idx[c] for c in text])

# --------------------------------
# 3. CREATE TRAINING SEQUENCES
# --------------------------------
seq_length = 100  # Length of each training example
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

# Set global batch size. Note: TPU strategy will split the global batch among replicas.
GLOBAL_BATCH_SIZE = 64 * strategy.num_replicas_in_sync  # e.g., 64 * 8 = 512

BUFFER_SIZE = 10000
dataset = dataset.shuffle(BUFFER_SIZE).batch(GLOBAL_BATCH_SIZE, drop_remainder=True)

# --------------------------------
# 4. BUILD THE TRAINING MODEL (non-stateful for TPU distribution)
# --------------------------------
with strategy.scope():
    inputs = tf.keras.layers.Input(shape=(None,), dtype=tf.int32)
    x = tf.keras.layers.Embedding(vocab_size, 256)(inputs)
    # Set stateful=False for training so that the model works with distributed batches
    x = tf.keras.layers.LSTM(1024,
                             return_sequences=True,
                             stateful=False,
                             recurrent_initializer='glorot_uniform')(x)
    outputs = tf.keras.layers.Dense(vocab_size)(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Define loss (from_logits=True because the Dense layer does not use an activation)
    def loss(labels, logits):
        return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

    model.compile(optimizer='adam', loss=loss)

model.summary()

# --------------------------------
# 5. TRAIN THE MODEL
# --------------------------------
EPOCHS = 20

history = model.fit(dataset, epochs=EPOCHS)

# --------------------------------
# 6. SAVE THE MODEL FOR LATER USE
# --------------------------------
model_save_path = "roman_urdu_poetry_model.keras"
model.save(model_save_path)
print("Model saved to", model_save_path)

# --------------------------------
# 7. BUILD THE GENERATION MODEL (with stateful LSTM and batch_size=1)
# --------------------------------
def build_generation_model(trained_model, batch_size=1):
    """
    Rebuild the model for text generation with a fixed batch size of 1.
    The generation model uses stateful LSTM so that it can maintain context
    across generated characters.
    """
    inputs = tf.keras.layers.Input(batch_shape=(batch_size, None), dtype=tf.int32)
    x = tf.keras.layers.Embedding(vocab_size, 256)(inputs)
    x = tf.keras.layers.LSTM(1024,
                             return_sequences=True,
                             stateful=True,
                             recurrent_initializer='glorot_uniform')(x)
    outputs = tf.keras.layers.Dense(vocab_size)(x)

    gen_model = tf.keras.Model(inputs=inputs, outputs=outputs)
    gen_model.set_weights(trained_model.get_weights())
    return gen_model

# --------------------------------
# 8. TEXT GENERATION FUNCTION
# --------------------------------
def generate_text(model, start_string, num_generate=1000, temperature=1.0):
    """
    Generates text using the trained model.

    Args:
      model: The trained generation model.
      start_string: Seed text to start generation.
      num_generate: Number of characters to generate.
      temperature: Controls randomness (lower values yield more predictable output).
    """
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []
    for layer in model.layers:
        if hasattr(layer, "reset_states"):
            layer.reset_states()


    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)  # remove the batch dimension
        predictions = predictions / temperature

        # Sample the next character from the distribution.
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()

        # Pass the predicted character as the next input.
        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx2char[predicted_id])

    return start_string + ''.join(text_generated)

# --------------------------------
# 9. GENERATE SAMPLE TEXT
# --------------------------------
# Build the generation model (with batch_size = 1 and stateful LSTM)
gen_model = build_generation_model(model, batch_size=1)

# Provide a seed text (can be any Roman‑Urdu string)
seed_text = "aañkh se "

generated_poetry = generate_text(gen_model, start_string=seed_text, num_generate=1000, temperature=0.8)
print("\nGenerated Poetry:\n")
print(generated_poetry)


TPU already initialized. Reusing the existing TPU strategy.
Number of replicas: 8
Length of text: 890281 characters
47 unique characters


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)           │ (None, None)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding_7 (Embedding)              │ (None, None, 256)           │          12,032 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_7 (LSTM)                        │ (None, None, 1024)          │       5,246,976 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, None, 47)            │          48,175 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,307,183 (20.25 MB)

 Trainable params: 5,307,183 (20.25 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 6s 80ms/step - loss: 3.8429
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - loss: 3.1084
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - loss: 2.9812
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 2.6198
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - loss: 2.4002
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 2.2940
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - loss: 2.1806
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 79ms/step - loss: 2.1311
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 2.0705
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 2.0188
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 1.9699
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 79ms/step - loss: 1.9160
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 1.8734
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - loss: 1.8538
Epoch 15/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 84ms/step - loss: 1.7961
Epo